
# Crowd-Sense — Real-Time Crowd Density Monitoring

This notebook runs the Crowd-Sense prototype on an uploaded video.

**Pipeline:** Video → Person detection (YOLO) → Zone-wise counting → Green/Yellow/Red heatmap → Danger alert

### What this version improves
- Works with a newly uploaded video each time.
- Processes the **whole video** instead of automatically cutting it to 20 seconds.
- Lets you change the number of rows/columns and thresholds from one configuration cell.
- Keeps the zone layout independent of video resolution.
- Shows a prominent `DANGER` banner when any zone crosses the red threshold.
- Reports the maximum number of people seen in any one zone.


In [1]:

# Cell 1 — Install dependencies
!pip -q install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.1/96.1 kB 5.1 MB/s eta 0:00:00


In [2]:

# Cell 2 — Upload ONE video
from google.colab import files
import os

uploaded = files.upload()

video_files = [
    name for name in uploaded.keys()
    if name.lower().endswith((".mp4", ".mov", ".mkv", ".avi", ".webm", ".3gp"))
]

if not video_files:
    raise ValueError("Please upload a video file such as MP4, MOV, AVI, MKV or WEBM.")

if len(video_files) > 1:
    print("Multiple videos uploaded. Using:", video_files[0])

INPUT_VIDEO = video_files[0]
print("Selected video:", INPUT_VIDEO)


Saving crowd.mp4 to crowd.mp4
Selected video: crowd.mp4


In [3]:

# Cell 3 — Configuration
# Change these values if you want a different zone layout or thresholds.

ROWS = 3
COLS = 2

YELLOW_THRESHOLD = 3
RED_THRESHOLD = 6

# YOLO settings
MODEL_NAME = "yolov8m.pt"
CONFIDENCE = 0.20
IMAGE_SIZE = 1280

# Output names
FIXED_VIDEO = "crowdsense_input.mp4"
OUTPUT_VIDEO = "crowdsense_output.mp4"
BEST_FRAME = "crowdsense_best_frame.jpg"

print(f"Zones: {ROWS} x {COLS} = {ROWS * COLS}")
print(f"Thresholds: Yellow >= {YELLOW_THRESHOLD}, Red >= {RED_THRESHOLD}")


Zones: 3 x 2 = 6
Thresholds: Yellow >= 3, Red >= 6


In [4]:

# Cell 4 — Convert the uploaded video to a broadly compatible MP4
# This keeps the full duration. No 20-second cutoff.

import subprocess
import cv2
import os

cmd = [
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", INPUT_VIDEO,
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-an",
    FIXED_VIDEO
]

result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("FFmpeg could not convert the uploaded video.")

test_cap = cv2.VideoCapture(FIXED_VIDEO)
if not test_cap.isOpened():
    raise RuntimeError("OpenCV could not open the converted video.")

fps = test_cap.get(cv2.CAP_PROP_FPS) or 25
width = int(test_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(test_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_count = int(test_cap.get(cv2.CAP_PROP_FRAME_COUNT))
duration = frame_count / fps if fps else 0
test_cap.release()

print(f"Video: {width} x {height}")
print(f"FPS: {fps:.2f}")
print(f"Frames: {frame_count}")
print(f"Duration: {duration:.1f} seconds")


Video: 478 x 850
FPS: 30.00
Frames: 320
Duration: 10.7 seconds


In [5]:

# Cell 5 — Crowd-Sense processing
from ultralytics import YOLO
import cv2
import numpy as np
import os

model = YOLO(MODEL_NAME)

def zone_status(count):
    if count >= RED_THRESHOLD:
        return "RED"
    elif count >= YELLOW_THRESHOLD:
        return "YELLOW"
    return "GREEN"

def status_color(status):
    if status == "RED":
        return (0, 0, 255)
    if status == "YELLOW":
        return (0, 255, 255)
    return (0, 200, 0)

cap = cv2.VideoCapture(FIXED_VIDEO)

if not cap.isOpened():
    raise RuntimeError("Could not open the prepared video.")

fps = cap.get(cv2.CAP_PROP_FPS) or 25
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

out = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (w, h)
)

frame_number = 0
max_seen_in_one_zone = 0
danger_frames = 0

while True:
    ok, frame = cap.read()
    if not ok:
        break

    counts = [[0] * COLS for _ in range(ROWS)]

    # Person class = 0 in COCO
    result = model(
        frame,
        classes=[0],
        imgsz=IMAGE_SIZE,
        conf=CONFIDENCE,
        verbose=False
    )[0]

    # Count a person in the zone containing the center of the bounding box.
    if result.boxes is not None:
        for box in result.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = box
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)

            r = min(max(cy * ROWS // h, 0), ROWS - 1)
            c = min(max(cx * COLS // w, 0), COLS - 1)
            counts[r][c] += 1

            cv2.circle(frame, (cx, cy), 5, (255, 255, 255), -1)

    zone_max = max(max(row) for row in counts)
    max_seen_in_one_zone = max(max_seen_in_one_zone, zone_max)

    if zone_max >= RED_THRESHOLD:
        danger_frames += 1

    # Draw the zone heatmap.
    overlay = frame.copy()

    for r in range(ROWS):
        for c in range(COLS):
            n = counts[r][c]
            status = zone_status(n)
            color = status_color(status)

            p1 = (c * w // COLS, r * h // ROWS)
            p2 = ((c + 1) * w // COLS, (r + 1) * h // ROWS)

            cv2.rectangle(overlay, p1, p2, color, -1)
            cv2.rectangle(frame, p1, p2, color, 2)

            label = f"Zone {r * COLS + c + 1}: {n}"
            if status == "RED":
                label += "  DANGER"

            cv2.putText(
                overlay,
                label,
                (p1[0] + 10, p1[1] + 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.75,
                (0, 0, 0),
                2,
                cv2.LINE_AA
            )

    frame = cv2.addWeighted(overlay, 0.35, frame, 0.65, 0)

    # Prominent alert banner.
    if zone_max >= RED_THRESHOLD:
        banner_text = "DANGER ZONE - CROWD THRESHOLD CROSSED"
        cv2.rectangle(frame, (0, 0), (w, 55), (0, 0, 255), -1)
        cv2.putText(
            frame, banner_text, (15, 38),
            cv2.FONT_HERSHEY_SIMPLEX, 0.85,
            (255, 255, 255), 2, cv2.LINE_AA
        )

    out.write(frame)
    frame_number += 1

cap.release()
out.release()

print("Processing complete.")
print("Frames processed:", frame_number)
print("Most people detected in one zone:", max_seen_in_one_zone)
print("Frames with a red/danger zone:", danger_frames)
print("Output video:", OUTPUT_VIDEO)


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Processing complete.
Frames processed: 320
Most people detected in one zone: 10
Frames with a red/danger zone: 266
Output video: crowdsense_output.mp4


In [6]:

# Cell 6 — Preview the result in Colab
from IPython.display import Video, display
from google.colab import files
import os

if os.path.exists(OUTPUT_VIDEO):
    display(Video(OUTPUT_VIDEO, embed=True, width=700))

print("Download the processed video:")
files.download(OUTPUT_VIDEO)


Download the processed video:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>